# Semana 3 — Satélite de diagnóstico: clasificación **multiclase**

> **Cuaderno satélite (ALUMNO).** Tiene **3 TODO** que completar. Complementa a `Semana3_clasificacion_ALUMNO/SOLUCION.ipynb`,
> que define el marco de evaluación sobre problemas **binarios**. Aquí damos el salto a **K clases**.

En el satélite de mecánica clasificamos vinos blancos como **buenos** o **malos** con la regla
`quality > 5`. Esa binarización era cómoda, pero **tiró información**: la variable `quality` del
dataset de UCI no es un sí/no, es una **puntuación de 3 a 9 dada por catadores**.

Esta es la pregunta de este cuaderno: *¿qué pasa cuando NO binarizamos?* Y la respuesta trae tres
lecciones que el caso binario no puede enseñar:

1. La **matriz de confusión** deja de ser 2×2 y pasa a ser K×K — y se lee distinto.
2. «F1» deja de ser un número: hay **tres medias** distintas (macro, weighted, micro) que aquí dan
   resultados radicalmente diferentes. Decir «F1 = 0,5» sin decir cuál es una cifra vacía.
3. Con clases muy raras aparece un problema nuevo: hay clases que el modelo **no predice nunca**,
   y las métricas globales lo esconden.

**Por qué importa**: en una planta real, el diagnóstico de fallos casi nunca es binario: es *qué*
está fallando, entre varias opciones. Este cuaderno prepara para ese tipo de problema.

## 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             confusion_matrix, classification_report)

SEMILLA = 0
np.set_printoptions(precision=3, suppress=True)

## 1. El dato: siete clases, no dos

Cargamos el mismo fichero de vinos blancos del satélite de mecánica, pero esta vez **sin tocar**
la columna `quality`.

In [ ]:
vinos = pd.read_csv("../datasets/winequality-white.csv", sep=";")
X = vinos.iloc[:, :-1].to_numpy()
y = vinos["quality"].to_numpy()

print(f"n = {len(y)} vinos, {X.shape[1]} caracteristicas fisico-quimicas")
print(f"clases presentes: {np.unique(y)}\n")

reparto = pd.Series(y).value_counts().sort_index()
for clase, cuenta in reparto.items():
    print(f"  quality={clase}: {cuenta:5d} vinos  ({cuenta/len(y):6.2%})  {'#'*int(60*cuenta/len(y))}")

Mira el reparto: **no es un desbalanceo moderado como el de la binarización buenos/malos**
(67 % / 33 %). Aquí hay
2.198 vinos de calidad 6 y **5 vinos de calidad 9**. Eso es un factor de 440×.

Y aquí está lo que la binarización `quality > 5` escondía: convertía siete clases muy desiguales en
dos clases casi equilibradas, y con ello **borraba el problema**. La comodidad tenía un precio.

## 2. La línea base, otra vez lo primero

Misma disciplina que en el resto del curso: antes de ningún modelo, **la línea base**. En multiclase
la línea base sigue siendo predecir siempre la clase mayoritaria — aquí, `quality = 6`.

Partición aleatoria 70/30. (Estos datos **no** son una serie temporal: cada vino es independiente,
así que la partición aleatoria es legítima.)

In [ ]:
rng = np.random.default_rng(SEMILLA)
idx = rng.permutation(len(y))
corte = int(0.7 * len(y))
tr, te = idx[:corte], idx[corte:]

escalador = StandardScaler().fit(X[tr])
Xtr, Xte = escalador.transform(X[tr]), escalador.transform(X[te])
ytr, yte = y[tr], y[te]
clases = np.unique(yte)

print(f"train = {len(tr)} | test = {len(te)}")
print(f"clases en test: {clases}")
print("soporte por clase en test:", {int(k): int((yte == k).sum()) for k in clases})

In [ ]:
mayoritaria = pd.Series(ytr).mode()[0]

# TODO 1: construye la prediccion de la linea base (predecir SIEMPRE la clase mayoritaria)
#   y mide su accuracy, su F1 macro y su F1 weighted.
#   Pista: np.full_like(yte, valor) crea un array del tamano de yte lleno de ese valor.
#   Ojo: usa zero_division=0 en f1_score, o veras avisos por las clases que nunca se predicen.
pred_base = ...   # <-- TODO
raise NotImplementedError('Completa el TODO 1')

print(f"Linea base: predecir siempre quality={mayoritaria}\n")
print(f"  accuracy    = {accuracy_score(yte, pred_base):.3f}")
print(f"  F1 macro    = {f1_score(yte, pred_base, average='macro', zero_division=0):.3f}")
print(f"  F1 weighted = {f1_score(yte, pred_base, average='weighted', zero_division=0):.3f}")

Primera trampa: **accuracy 0,438 sin haber aprendido nada**. Como la clase 6 es casi la mitad del
dataset, acertar siempre 6 ya «acierta» el 44 % de las veces.

Fíjate en la distancia entre las dos F1: **macro 0,087 frente a weighted 0,267**. Todavía no hemos
explicado por qué difieren — ese es el apartado 4 — pero anota ya que un mismo modelo, con las
mismas predicciones, produce dos cifras que se llevan un factor de 3.

## 3. Regresión logística multinomial y la matriz K×K

`LogisticRegression` de scikit-learn resuelve el caso multiclase directamente (ajusta un vector de
coeficientes por clase y normaliza con *softmax*). No hay que hacer nada especial.

> **Nota de versión**: en scikit-learn ≥ 1.7 el parámetro `multi_class` fue **eliminado**; el
> comportamiento multinomial es el de por defecto. Si ves código antiguo con
> `LogisticRegression(multi_class='multinomial')`, hoy da error.

In [ ]:
logistica = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
pred_lg = logistica.predict(Xte)

print(f"accuracy de la logistica = {accuracy_score(yte, pred_lg):.3f}")
print(f"(la linea base estaba en   {accuracy_score(yte, pred_base):.3f})")

In [ ]:
cm = confusion_matrix(yte, pred_lg, labels=clases)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(clases)), clases)
ax.set_yticks(range(len(clases)), clases)
ax.set_xlabel("clase predicha"); ax.set_ylabel("clase real")
ax.set_title("Matriz de confusion 7x7 - logistica multinomial")
for i in range(len(clases)):
    for j in range(len(clases)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout(); plt.show()

print(cm)

Léela por **columnas**, que es donde está la sorpresa: las columnas de las clases **3, 8 y 9 están
enteras a cero**. El modelo no predice esas clases **jamás**, para ningún vino del test.

No es un fallo de programación: con 20 vinos de calidad 3 y 5 de calidad 9 en todo el dataset, a un
modelo que minimiza el error global le sale rentable no apostar nunca por ellas. Y la accuracy, que
es un promedio sobre vinos, **no se entera**.

## 4. «F1» no es un número: son tres

Con dos clases, F1 era una cifra. Con K clases hay que **promediar** los F1 de cada clase, y hay
tres formas de hacerlo:

| Media | Cómo promedia | A quién favorece |
|---|---|---|
| **macro** | media aritmética simple de los F1 por clase | trata todas las clases **igual**: las raras pesan lo mismo que las frecuentes |
| **weighted** | media ponderada por el número de ejemplos de cada clase | las clases frecuentes **dominan** el resultado |
| **micro** | agrega aciertos y fallos de todas las clases antes de calcular | en clasificación multiclase de etiqueta única, **coincide exactamente con la accuracy** |

In [ ]:
# TODO 2: calcula el F1 con las TRES medias ("macro", "weighted", "micro") y tambien la accuracy.
#   Fijate bien en las tres cifras antes de seguir: una de ellas coincide exactamente con
#   la accuracy. ?Cual, y por que? Lo comentamos en la celda de texto siguiente.
raise NotImplementedError('Completa el TODO 2')

for media in ("macro", "weighted", "micro"):
    print(f"  F1 {media:<9} = {...:.3f}")   # <-- TODO: sustituye ... por la llamada a f1_score
print(f"\n  accuracy      = {...:.3f}")     # <-- TODO

Tres cifras para el mismo modelo y las mismas predicciones: **0,214 · 0,473 · 0,516**.

- **macro = 0,214** es la más dura y la más honesta aquí: promedia las siete clases por igual, así
  que las tres clases con F1 = 0 (las que el modelo nunca predice) arrastran la media hacia abajo.
- **weighted = 0,473** es más de el doble, porque pondera por soporte: las clases 5 y 6, que son el
  73 % del test y las únicas que el modelo predice razonablemente, se llevan casi todo el peso.
- **micro = 0,516** coincide con la accuracy, exactamente. No es casualidad: con etiqueta única,
  cada vino aporta o un acierto o un fallo, así que agregarlos antes de dividir es literalmente
  contar aciertos.

**La lección**: publicar «F1 = 0,5» sin decir qué media es, en un problema multiclase desbalanceado,
es publicar una cifra sin significado. Y si alguien elige *weighted* sin justificarlo, está eligiendo
la media que mejor le hace quedar.

## 5. El recall por clase: dónde se ve lo que las medias esconden

La forma de no dejarse engañar es sencilla: **mirar clase por clase** antes de mirar cualquier
media.

In [ ]:
# TODO 3: calcula el recall de CADA clase por separado y muestralo junto a su soporte
#   (cuantos vinos de esa clase hay en test). Marca las clases cuyo recall sea 0.
#   Pista: recall_score(..., average=None, labels=clases, zero_division=0) devuelve un
#   array con un recall por clase, en el orden de `clases`.
recalls = ...   # <-- TODO
raise NotImplementedError('Completa el TODO 3')

print(f"{'clase':>6} {'soporte':>8} {'recall':>8}")
for c, r in zip(clases, recalls):
    n_c = int((yte == c).sum())
    aviso = "   <-- nunca predicha" if r == 0 else ""
    print(f"{c:>6} {n_c:>8} {r:>8.3f}{aviso}")

In [ ]:
print(classification_report(yte, pred_lg, zero_division=0))

Aquí está el diagnóstico completo, y es incómodo: de siete clases, el modelo solo tiene recall
apreciable en **dos** (la 5 con 0,53 y la 6 con 0,75). La clase 7 se queda en 0,17 y las clases 3, 8
y 9 en **cero absoluto**.

Ante esto hay tres salidas honestas, y elegir es parte del trabajo del ingeniero:

1. **Aceptarlo y declararlo**: el modelo sirve para distinguir vinos mediocres de normales, y no
   sirve para detectar los excelentes ni los pésimos. Es una limitación, no un secreto.
2. **Agrupar clases**: fundir {3,4} en «malo» y {8,9} en «excelente» da clases con suficiente
   soporte. Es exactamente lo que hacía la binarización del satélite de mecánica, pero de forma
   consciente y menos drástica.
3. **Reponderar** (`class_weight='balanced'`): fuerza al modelo a atender a las clases raras. Sube
   su recall, pero baja la accuracy global y dispara las falsas alarmas: el compromiso de coste
   asimétrico de siempre.

## 6. Random Forest: ¿gana un ensemble?

¿Bate un Random Forest a la regresión logística? Que un ensemble gane o no depende del problema, así
que no conviene suponerlo en ningún sentido. Probemos aquí.

In [ ]:
bosque = RandomForestClassifier(n_estimators=300, random_state=SEMILLA).fit(Xtr, ytr)
pred_rf = bosque.predict(Xte)

print(f"{'metrica':<22} {'logistica':>11} {'random forest':>15}")
print("-" * 50)
print(f"{'accuracy':<22} {accuracy_score(yte, pred_lg):>11.3f} {accuracy_score(yte, pred_rf):>15.3f}")
print(f"{'F1 macro':<22} {f1_score(yte, pred_lg, average='macro', zero_division=0):>11.3f} {f1_score(yte, pred_rf, average='macro', zero_division=0):>15.3f}")
print(f"{'F1 weighted':<22} {f1_score(yte, pred_lg, average='weighted', zero_division=0):>11.3f} {f1_score(yte, pred_rf, average='weighted', zero_division=0):>15.3f}")

print("\nrecall por clase (random forest):")
for c, r in zip(clases, recall_score(yte, pred_rf, average=None, labels=clases, zero_division=0)):
    print(f"   quality={c}: {r:.3f}")

Aquí el Random Forest **gana con claridad**: accuracy 0,688 frente a 0,516, F1 macro casi el doble
(0,403 frente a 0,214). Además **rescata parcialmente la clase 8**
(recall 0,404, cuando la logística la ignoraba por completo).

¿Por qué gana aquí? Porque un modelo flexible solo aporta si hay estructura que el lineal no puede
representar:

- **Aquí**, la relación entre química y puntuación del catador es **claramente no lineal y con
  interacciones** (el alcohol importa distinto según la acidez volátil, etc.), que es justo lo que
  un bosque captura y un modelo lineal no puede.
- Si la relación fuera **esencialmente lineal y débil**, un modelo flexible no encontraría más
  estructura, solo más ruido con el que sobreentrenarse, y la logística empataría o ganaría.

**La conclusión que hay que llevarse** no es «los ensembles ganan» ni «los ensembles no ganan», sino
que **eso se mide, no se supone** — en cada problema, contra una línea base, con la métrica adecuada.

## 7. Cambio de dominio: el mismo modelo, otro vino

Un último experimento, que es un anticipo directo de la Semana 6. Nuestro modelo está entrenado con
vinos **blancos**. ¿Qué pasa si lo aplicamos a vinos **tintos**, que tienen exactamente las mismas
once variables físico-químicas medidas de la misma forma?

In [ ]:
tintos = pd.read_csv("../datasets/winequality-red.csv", sep=";")
Xr = escalador.transform(tintos.iloc[:, :-1].to_numpy())   # OJO: escalador ajustado en BLANCOS
yr = tintos["quality"].to_numpy()

print(f"vinos tintos: {len(yr)}\n")
print(f"{'modelo':<16} {'acc blancos':>12} {'acc tintos':>12}")
print("-" * 42)
for nombre, modelo, pred_blanco in (
        ("logistica", logistica, pred_lg),
        ("random forest", bosque, pred_rf)):
    pr = modelo.predict(Xr)
    print(f"{nombre:<16} {accuracy_score(yte, pred_blanco):>12.3f} {accuracy_score(yr, pr):>12.3f}")

El desplome es brutal: la logística cae de **0,516 a 0,153** y el bosque de **0,688 a 0,353**.

Y sin embargo **no ha cambiado nada del código**: mismas variables, mismas unidades, mismo pipeline.
Lo único que ha cambiado es que los tintos tienen otra distribución (más acidez volátil, menos
azúcar residual, otro reparto de calidades) y la relación entre química y puntuación **no es la misma**.

Esto tiene nombre y lo verás formalmente en la **Semana 6**: es *dataset shift*. Un modelo solo es
válido dentro del dominio del que aprendió, y **nada en las métricas de test te avisa** de que lo
estás sacando de ahí — el test de blancos seguía diciendo 0,516. Por eso la S6 monitoriza el proceso
(MSPC) en vez de fiarse del rendimiento histórico.

## 8. Resumen

Lo que este cuaderno añade al marco de evaluación del satélite de mecánica:

| Concepto | En binario | En multiclase |
|---|---|---|
| Matriz de confusión | 2×2, cuatro casillas con nombre propio | K×K; se lee por columnas para ver qué clases **nunca se predicen** |
| F1 | un número | **tres** (macro / weighted / micro); hay que decir cuál y por qué |
| Clases raras | «clase minoritaria» | clases que el modelo ignora por completo, invisibles en las medias globales |

**La disciplina no cambia**: línea base primero, métricas honestas, y la pregunta de siempre:
*¿qué error cuesta dinero?* Lo que cambia es que ahora tienes herramientas para cuando el
diagnóstico industrial no sea un sí/no, que es el caso habitual en la detección de fallos reales.